# Yoruba Acoustic Model — Kaggle Training

Trains the MFA acoustic model on a **merged, multi-source** corpus, in an
environment with more disk, cores and time than a laptop.

This notebook only does the training half: fetch → merge → validate → train →
export. Analysis, tone work and scoring stay in `train_mfa_yoruba.ipynb`, which
runs locally against the model this produces.

## Before you start

1. **Settings → Internet: On** (required for downloads).
2. **Settings → Accelerator: None.** MFA is a Kaldi GMM-HMM pipeline with no
   CUDA path whatsoever, so a GPU does nothing here except consume your GPU
   quota. Pick the option with the most CPU cores.
3. Kaggle CPU sessions run up to 12 hours and `/kaggle/working` holds 20 GB.
   Both are real constraints here — see the budget section.

## The sources, and what each is actually good for

| Source | Format | Tone marks | Verdict |
|---|---|---|---|
| **OpenSLR SLR86** | 48 kHz WAV, sentences | ~100% | Best quality. Tone reference. |
| **michsethowusu/yoruba-speech-text-parallel** | 16 kHz WAV, 1–2 word clips | ~84% | Huge; isolated words, not sentences. |
| **google/WaxalNLP `yor_tts`** | 48 kHz MP3, 10–20 s | ~32% | Acoustic training only. |
| **Yoruba-TTS-Dataset** (yours) | MP3 | 0% | Acoustic training only. |

Two findings worth knowing before you spend a session on this:

**OpenSLR SLR86 is the corpus you already have.** `yo_ng_female` and
`yo_ng_male` *are* SLR86. Re-downloading it on Kaggle adds nothing unless you
want everything in one place.

**Only SLR86 can serve as a tone reference.** Tone comparison needs diacritics.
WaxalNLP at ~32% and the TTS dataset at 0% cannot supply them, and mixing
undiacritized text into a tone-suffixed dictionary would label their vowels Mid
and corrupt the phone models. They are included for acoustic coverage only, and
this notebook forces tone-stripped phones whenever they are enabled.

In [1]:
!pip install praat-parselmouth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 82.3 MB/s eta 0:00:00


---
## 1. Configuration

Set what you want, then run the notebook top to bottom.


In [2]:
from pathlib import Path

# --- Which sources to include -------------------------------------------------
USE_OPENSLR      = True    # SLR86: the diacritized sentence corpus
USE_MICHSETHOWUSU= True    # 1-2 word clips, ~84% tone-marked
USE_WAXAL        = True   # long MP3, ~32% tone-marked; acoustic coverage only
USE_LOCAL_TTS    = True   # your Yoruba-TTS-Dataset, undiacritized

# Point this at the Yoruba-TTS-Dataset once you have added it as a Kaggle
# dataset. The exact name appears under /kaggle/input after you attach it.
# Kaggle nests attached datasets under /kaggle/input/datasets/<user>/<name>/.
LOCAL_TTS_DIR = Path("/kaggle/input/datasets/pearsejim01/yoruba-tts-dataset/Yoruba-TTS-Dataset")


# --- Size limits --------------------------------------------------------------
# Verified against the Hub: michsethowusu is 50 shards x ~420 MB (21.5 GB,
# ~1.65M clips); WaxalNLP data/TTS/yor is 11 files x 90-913 MB (2.6 GB).
# One michsethowusu shard already yields ~2.4 h of usable audio after filtering,
# so a couple of shards is plenty -- the limit here is training time, not data.
# How many shards to DOWNLOAD. This is about disk and fetch time, and is a
# separate question from how much audio to TRAIN on (MAX_HOURS below).
#
# Download broadly, then sample across everything fetched: taking 2 h of audio
# spread over 12 shards gives more speakers and recording sessions than 2 h
# taken from the first shard, and variety is what an acoustic model needs.
#
# None means "every shard". michsethowusu in full is 21.5 GB, which fits Kaggle
# disk but costs fetch time; 12 shards is a good balance.
MICHSETHOWUSU_SHARDS = None
WAXAL_SHARDS         = None      # all 11 files, 2.6 GB

# Disk guard, in GB per source. None for no limit.
MAX_DOWNLOAD_GB = {"michsethowusu": 30.0, "waxal": None}

# Which splits to pull from the Hugging Face datasets.
#
# A dataset's test/validation splits exist to protect its authors' benchmarks.
# We report no numbers on those benchmarks, and acoustic training is EM
# alignment of audio against known transcripts rather than supervised
# classification, so their splits are simply more audio to us.
#
# Our own evaluation is kept honest a different way: whole SPEAKERS are held out
# from SLR86 below and never trained on. See HOLDOUT_SPEAKER_FRACTION.
HF_SPLITS = ("train", "validation", "test")

# Hours of audio per source to actually TRAIN on, applied after filtering, and
# sampled at random across every shard downloaded rather than taken from the
# front. Two purposes: keeping the run inside the session limit, and stopping
# one source swamping the others -- a model trained mostly on isolated words
# aligns connected speech worse.
# MAX_HOURS = {"openslr": 6.0, "michsethowusu": 24.0, "waxal": 12.0, "local_tts": 6.0}
MAX_HOURS = {"openslr": 6.0, "michsethowusu": 24.0, "waxal": 12.0, "local_tts": 6.0}




# --- Filters ------------------------------------------------------------------
# Measured on michsethowusu with the current model: clips >= 0.3 s aligned
# 18/18, clips < 0.3 s only 27/42. The cutoff is empirical, not a guess.
MIN_DURATION_S = 0.30
MAX_DURATION_S = 15.0   # above this alignment gets slow and unreliable
REQUIRE_TONE_MARKS = False  # True keeps only diacritized lines (SLR86 only)

# --- Speaker labelling --------------------------------------------------------
# michsethowusu is crowdsourced and carries no speaker field. Left alone, every
# clip lands in one directory, which breaks two things that are keyed on the
# speaker: MFA's speaker-adaptive stage estimates one transform for tens of
# thousands of different people, and the vowel classifier's cepstral
# normalization degenerates to a global mean that removes nothing.
#
# Bucketing gives both something usable. 50 clips of 1-2 words is roughly a
# minute of audio -- thin for an fMLLR transform, but far better than one
# transform shared across the whole source.
BASE_BUCKET_SIZE = 50

# --- Held-out evaluation ------------------------------------------------------
# A fraction of SLR86 speakers is withheld from training entirely and aligned
# afterwards, giving an out-of-sample measure of how well the model handles a
# voice it has never heard.
#
# The split is by SPEAKER, not by utterance. MFA estimates a per-speaker
# transform during its speaker-adaptive stage, so holding out random utterances
# from speakers it trained on would flatter the result.
#
# SLR86 is used because it is the only fully diacritized source, so tone can be
# checked on it too -- not just alignment success.
HOLDOUT_SPEAKER_FRACTION = 0.15

# --- Paths --------------------------------------------------------------------
WORK    = Path("/kaggle/working")
TEMP    = Path("/kaggle/temp")          # not counted against the 20 GB output
RAW     = TEMP / "raw"
CORPUS  = TEMP / "mfa_corpus"
OUTPUT  = WORK / "yoruba_model"

for d in (RAW, CORPUS, OUTPUT):
    d.mkdir(parents=True, exist_ok=True)

import os
NUM_JOBS = os.cpu_count() or 4
print(f"CPUs: {NUM_JOBS}")
print(f"Sources: openslr={USE_OPENSLR} michsethowusu={USE_MICHSETHOWUSU} "
      f"waxal={USE_WAXAL} local_tts={USE_LOCAL_TTS}")

# --- Training-time budget ----------------------------------------------------
# Measured runs, not extrapolation:
#
#   4 h audio,  3,583 utterances, 48 kHz source, 16 local cores -> 3 h 23 min
#   18 h audio, mostly short 16 kHz clips,       Kaggle          -> 2 h 30 min
#
# The second is 6x faster per hour of audio on fewer cores, so hours of audio is
# clearly not the thing that governs runtime. What does:
#
#   * Sample-rate conversion. MFA resamples everything to 16 kHz; a 48 kHz
#     corpus pays that cost on every file, every pass. This notebook writes
#     16 kHz WAVs up front, which the local run did not.
#   * Utterance count and speaker count, through graph compilation and the
#     per-speaker transforms in the SAT stage.
#   * Whether the machine is otherwise busy. The local figure was measured on a
#     laptop running other work, and is best treated as a bad case.
#
# The estimate below therefore uses the Kaggle measurement and is deliberately
# labelled rough. Watch the elapsed-time output of the training cell rather than
# trusting it.
enabled = [
    ("openslr", USE_OPENSLR), ("michsethowusu", USE_MICHSETHOWUSU),
    ("waxal", USE_WAXAL), ("local_tts", USE_LOCAL_TTS),
]
budget_hours = sum(MAX_HOURS[name] for name, on in enabled if on)

# 18 h took 2.5 h on Kaggle with 16 kHz input: about 0.14 h of training per hour
# of audio. Scaled by cores against that run's core count.
KAGGLE_HOURS_PER_AUDIO_HOUR = 0.14
KAGGLE_CORES = 4
estimated_hours = budget_hours * KAGGLE_HOURS_PER_AUDIO_HOUR * (KAGGLE_CORES / NUM_JOBS)

print(f"\nAudio budget      : {budget_hours:.1f} h across "
      f"{sum(1 for _, on in enabled if on)} sources")
print(f"Rough estimate    : {estimated_hours:.1f} h on {NUM_JOBS} cores "
      f"(measured: 18 h audio -> 2.5 h)")

if estimated_hours > 9:
    print(
        "\nThis may not finish inside a 12-hour session, and an interrupted run "
        "saves no model. Consider lowering MAX_HOURS, but treat the estimate as "
        "indicative only -- runtime depends more on utterance count and sample "
        "rate than on hours of audio."
    )

CPUs: 4
Sources: openslr=True michsethowusu=True waxal=True local_tts=True

Audio budget      : 48.0 h across 4 sources
Rough estimate    : 6.7 h on 4 cores (measured: 18 h audio -> 2.5 h)


### Time and disk budget

MFA's speaker-adaptive stage dominates and scales roughly linearly with audio.
Measured on the reference machine: **4 hours of audio took 3 h 23 min on 15
cores.** Kaggle gives 4 cores, so expect roughly:

Two measured runs, which disagree sharply with any simple scaling rule:

| Corpus | Machine | Result |
|---|---|---|
| 4 h audio, 3,583 utterances, 48 kHz source | 16 local cores, busy laptop | 3 h 23 min |
| 18 h audio, mostly short 16 kHz clips | Kaggle | **2 h 30 min** |

The larger corpus finished faster on fewer cores. Hours of audio is not what
governs runtime — sample-rate conversion, utterance count and speaker count are.
This notebook writes 16 kHz WAVs up front, which removes the resampling cost the
local run paid on every pass.

Practically: 18 h of audio fits comfortably in a Kaggle session, so the earlier
advice to stay under 5 h was far too conservative. Watch the elapsed time the
training cell prints rather than trusting any estimate.

**Keep the merged corpus at or under about 5 hours** unless you plan to train
monophone-only. The caps in `MAX_HOURS` above total 15 hours if you enable
everything, which is deliberately more than you should use at once — trim them.

Disk: raw downloads land in `/kaggle/temp`, which does not count against the
20 GB `/kaggle/working` quota. Only the finished model and dictionary are
written to `/kaggle/working`.

In [3]:
# Install MFA. Micromamba is a single static binary and needs no shell setup,
# which suits a notebook environment better than a full conda install.
import subprocess, sys, os

MAMBA = Path("/kaggle/working/micromamba")
MFA_ENV = TEMP / "mfa-env"
MFA_BIN = MFA_ENV / "bin" / "mfa"

if not MFA_BIN.exists():
    if not MAMBA.exists():
        subprocess.run(
            "curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest "
            "| tar -xj -C /kaggle/working bin/micromamba && "
            "mv /kaggle/working/bin/micromamba /kaggle/working/micromamba",
            shell=True, check=True,
        )
    print("Installing Montreal Forced Aligner (a few minutes)...")
    subprocess.run(
        [str(MAMBA), "create", "-y", "-p", str(MFA_ENV),
         "-c", "conda-forge", "python=3.11", "montreal-forced-aligner"],
        check=True, env={**os.environ, "MAMBA_ROOT_PREFIX": "/kaggle/temp/mamba"},
    )

# MFA shells out to OpenFST tools that live beside the binary, so its bin
# directory must be on PATH -- calling mfa by absolute path is not enough.
os.environ["PATH"] = f"{MFA_ENV / 'bin'}{os.pathsep}{os.environ['PATH']}"
# MFA_ROOT_DIR is chosen in the next cell, after measuring the filesystems.

print(subprocess.run([str(MFA_BIN), "version"], capture_output=True, text=True).stdout.strip())

Installing Montreal Forced Aligner (a few minutes)...
Fetch Shard Index for conda-forge/linux-64                                                      ⧖ Starting
Fetch Shard Index for conda-forge/linux-64                                                ✔ Done (0.1 sec)
Fetch Shard Index for conda-forge/noarch                                                        ⧖ Starting
Fetch Shard Index for conda-forge/noarch                                                  ✔ Done (0.1 sec)
Fetching and Parsing Packages' Shards                                                           ⧖ Starting
Fetching and Parsing Packages' Shards                                                    ✔ Done (18.1 sec)

Resolving Environment                                                                           ⧖ Starting
Resolving Environment                                                                     ✔ Done (1.2 sec)

Transaction

  Prefix: /kaggle/temp/mfa-env

  Updating specs:

   - python=3.11
   - mo

warning  libmamba Security Warning: This transaction includes executing package scripts (pre/post-link/unlink) if present. These scripts can contain arbitrary code. Please ensure you trust the package sources.
warning  libmamba Executing post-link script for package 'gdk-pixbuf'.
warning  libmamba Executing post-link script for package 'gtk3'.
warning  libmamba Executing post-link script for package 'librsvg'.
warning  libmamba Executing post-link script for package 'graphviz'.


3.4.1


### Fix MFA's SQLite locking

MFA never sets a busy timeout on its SQLite connections, so they use the 5-second
default. Under four parallel workers that is exceeded during the final
`analyze_alignments()` step -- which runs *before* the model is exported, so the
crash costs the whole run.


In [4]:
# --- Fix MFA's SQLite locking ------------------------------------------------
#
# Two runs died after 4-5 hours with:
#   sqlite3.OperationalError: database is locked
# raised inside analyze_alignments(), which MFA calls from finalize_training()
# -- that is, BEFORE export_model(). The model is written after that call, so a
# crash there destroys the entire run's output.
#
# Cause: MFA builds its connection string as `sqlite:///{path}` with no connect
# arguments (montreal_forced_aligner/abc.py), so every connection inherits
# sqlite3's default 5-second busy timeout. Verified by experiment: with the
# default the write fails at exactly 5.0 s, with `?timeout=8` at exactly 8.0 s.
# Four concurrent workers writing phone intervals exceed 5 s easily.
#
# Fix: a sitecustomize module in MFA's environment. Python imports it
# automatically at interpreter start, so it applies to MFA's worker
# subprocesses too, which a monkey-patch inside the notebook would not.
#
#   busy_timeout=600000   wait up to 10 minutes for a lock instead of 5 seconds
#   journal_mode=WAL      readers no longer block writers, so contention drops
#   synchronous=NORMAL    fewer fsyncs; safe here since the database is scratch
#
# This changes only MFA's private database behaviour. It touches no data, no
# paths, and nothing in the engine.
site_packages = next(MFA_ENV.glob("lib/python*/site-packages"))
sitecustomize = site_packages / "sitecustomize.py"

sitecustomize.write_text(
    "import sqlite3\n"
    "from sqlalchemy import event\n"
    "from sqlalchemy.engine import Engine\n"
    "\n"
    "@event.listens_for(Engine, 'connect')\n"
    "def _sqlite_pragmas(dbapi_connection, connection_record):\n"
    "    if isinstance(dbapi_connection, sqlite3.Connection):\n"
    "        cursor = dbapi_connection.cursor()\n"
    "        cursor.execute('PRAGMA journal_mode=WAL')\n"
    "        cursor.execute('PRAGMA busy_timeout=600000')\n"
    "        cursor.execute('PRAGMA synchronous=NORMAL')\n"
    "        cursor.close()\n",
    encoding="utf-8",
)

check = subprocess.run(
    [str(MFA_ENV / "bin" / "python"), "-c",
     "import sqlalchemy, tempfile, pathlib;"
     "d=pathlib.Path(tempfile.mkdtemp())/'t.db';"
     "e=sqlalchemy.create_engine(f'sqlite:///{d}');"
     "c=e.connect();"
     "print(c.exec_driver_sql('PRAGMA journal_mode').scalar(),"
     "c.exec_driver_sql('PRAGMA busy_timeout').scalar())"],
    capture_output=True, text=True,
)
print(f"wrote {sitecustomize}")
print(f"verification (expect 'wal 600000'): {check.stdout.strip() or check.stderr.strip()[:200]}")

if "wal 600000" not in check.stdout:
    print("\nWARNING: the PRAGMAs did not take effect. Set USE_POSTGRES = True "
          "in the workspace cell before training, or the run may die at the "
          "final step after hours of work.")


wrote /kaggle/temp/mfa-env/lib/python3.11/site-packages/sitecustomize.py
verification (expect 'wal 600000'): wal 600000


In [5]:
# MFA workspace: SQLite database, feature archives, alignment lattices.
#
# This is SEPARATE from your data. MFA_ROOT_DIR controls only MFA's own scratch
# area (corpus.db, global_config.yaml, lattices). Downloads still go to RAW, the
# corpus to CORPUS, results to OUTPUT -- all of which stay on /kaggle/temp and
# are unaffected by this setting. MFA reads the corpus from the path passed as
# an argument and writes TextGrids to the output path passed as an argument.
#
# Why it matters: a run failed after 5.5 hours with
#   sqlite3.OperationalError: database is locked
# raised inside analyze_alignments(), which runs *before* the model is exported
# -- so the model was lost. SQLite locking is unreliable on network-backed and
# overlay filesystems, which is what Kaggle's working volumes are.
#
# The chosen directory is measured rather than assumed, because /tmp and
# /kaggle/temp may sit on the same device, in which case moving between them
# buys nothing and the real fix is USE_POSTGRES below.
import os
import shutil
import subprocess

def describe(path: Path) -> dict:
    """Filesystem type, device and free space for a candidate directory."""
    path.mkdir(parents=True, exist_ok=True)
    usage = shutil.disk_usage(path)
    try:
        fstype = subprocess.run(["stat", "-f", "-c", "%T", str(path)],
                                capture_output=True, text=True).stdout.strip()
    except Exception:
        fstype = "?"
    return {
        "path": str(path),
        "fstype": fstype or "?",
        "device": os.stat(path).st_dev,
        "free_gb": round(usage.free / 1e9, 1),
    }

candidates = [Path("/tmp/mfa_root"), TEMP / "mfa_root", WORK / "mfa_root"]
described = [describe(c) for c in candidates]

print(f"{'path':<28}{'fstype':<12}{'device':>10}{'free GB':>10}")
for d in described:
    print(f"{d['path']:<28}{d['fstype']:<12}{d['device']:>10}{d['free_gb']:>10}")

# MFA needs room for features and lattices: budget several GB for a large corpus.
usable = [d for d in described if d["free_gb"] >= 10]
chosen = usable[0] if usable else max(described, key=lambda d: d["free_gb"])

devices = {d["device"] for d in described}
if len(devices) == 1:
    print("\nNOTE: every candidate is on the same device, so relocating the MFA "
          "database cannot fix a filesystem-level locking problem. If the run "
          "fails with 'database is locked', set USE_POSTGRES = True below.")

os.environ["MFA_ROOT_DIR"] = chosen["path"]
print(f"\nMFA workspace : {chosen['path']}  ({chosen['free_gb']} GB free)")
print(f"Data (unchanged): raw={RAW}  corpus={CORPUS}  output={OUTPUT}")

# Postgres removes the SQLite locking failure mode rather than reducing its
# odds. It costs setup time, so it is off by default -- turn it on if a run has
# already died with 'database is locked'.
USE_POSTGRES = False


path                        fstype          device   free GB
/tmp/mfa_root               overlayfs           53    1099.9
/kaggle/temp/mfa_root       overlayfs           53    1099.9
/kaggle/working/mfa_root    ext2/ext3         1793      20.9

MFA workspace : /tmp/mfa_root  (1099.9 GB free)
Data (unchanged): raw=/kaggle/temp/raw  corpus=/kaggle/temp/mfa_corpus  output=/kaggle/working/yoruba_model


In [6]:
# The engine supplies the Yoruba grapheme-to-phoneme rules used to build the
# pronunciation dictionary. It is pure Python, so it needs no installing -- it
# only has to be importable as `yoruba_engine`.
#
# Kaggle complicates that in two ways: uploads are nested inside a folder named
# after the dataset, and dataset names use hyphens. A package uploaded as
# "yoruba-engine" therefore lands in a directory Python cannot import, because
# `yoruba-engine` is not a valid module name. So the package is located by its
# *contents*, then staged under the correct name.
PROJECT_GIT = ""   # e.g. "https://github.com/you/yoruba-engine.git"

import shutil, subprocess, sys

def looks_like_engine(directory: Path) -> bool:
    """Whether this directory is the yoruba_engine package itself."""
    return (
        (directory / "__init__.py").exists()
        and (directory / "g2p").is_dir()
        and (directory / "datatypes.py").exists()
    )


def find_engine_package(*roots: Path) -> Path | None:
    """Find the package directory, whatever it has been named."""
    for root in roots:
        if not root or not root.exists():
            continue
        if looks_like_engine(root):
            return root
        for candidate in root.rglob("*"):
            try:
                if candidate.is_dir() and looks_like_engine(candidate):
                    return candidate
            except PermissionError:
                continue
    return None

search_path = "/kaggle/input/datasets/pearsejim01/"
package = find_engine_package(Path(f"{search_path}"))

if package is None and PROJECT_GIT:
    clone = TEMP / "yoruba_engine_repo"
    if not clone.exists():
        subprocess.run(["git", "clone", "--depth", "1", PROJECT_GIT, str(clone)], check=True)
    package = find_engine_package(clone)

if package is None:
    print("Could not find the engine package. Under /kaggle/input:\n")
    for entry in sorted(Path(f"{search_path}").rglob("*")):
        depth = len(entry.relative_to(f"{search_path}").parts)
        if depth <= 3 and entry.is_dir():
            print("  " * depth + entry.name + "/")
    raise SystemExit(
        "\nThe upload must contain the engine package: a directory holding "
        "__init__.py, datatypes.py and g2p/."
    )

# /kaggle/input is read-only and the directory may be misnamed, so stage a copy
# under the importable name.
staged_root = TEMP / "engine"
staged = staged_root / "yoruba_engine"
if staged.exists():
    shutil.rmtree(staged)
staged.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(package, staged)

sys.path.insert(0, str(staged_root))
print(f"found package at : {package}")
print(f"staged as        : {staged}")

from yoruba_engine.g2p.dictionary import PronunciationDictionary, grapheme_to_phonemes
from yoruba_engine.g2p.orthography import has_tone_marks, normalize, tokenize

# Fail now, loudly, rather than after building 3,000 wrong dictionary entries.
assert [p.symbol for p in grapheme_to_phonemes("òrí")] == ["o_L", "r", "i_H"]
print("engine ready")


found package at : /kaggle/input/datasets/pearsejim01/yoruba-engine
staged as        : /kaggle/temp/engine/yoruba_engine
engine ready


---
## 2. Fetch the sources

Each downloader is independent and skips work already done, so a re-run after a
session restart resumes rather than starting over.

In [7]:
import csv, io, tarfile, urllib.request

def download(url: str, destination: Path) -> Path:
    'Download a URL to a path, skipping if it already exists.'
    if destination.exists() and destination.stat().st_size > 0:
        print(f"  have {destination.name}")
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    print(f"  fetching {url}")
    urllib.request.urlretrieve(url, destination)
    print(f"  {destination.stat().st_size / 1e6:.0f} MB")
    return destination


def duration_of(raw_bytes: bytes) -> float | None:
    'Duration in seconds of an in-memory audio file.'
    import soundfile as sf
    try:
        return float(sf.info(io.BytesIO(raw_bytes)).duration)
    except Exception:
        return None

In [8]:
# --- OpenSLR SLR86 -----------------------------------------------------------
# Crowdsourced Yoruba speech: the same data as yo_ng_female / yo_ng_male.
OPENSLR_FILES = [
    "https://www.openslr.org/resources/86/yo_ng_female.zip",
    "https://www.openslr.org/resources/86/yo_ng_male.zip",
]

def fetch_openslr() -> list[Path]:
    'Download and unzip SLR86, returning the extracted directories.'
    import zipfile
    directories = []
    for url in OPENSLR_FILES:
        name = url.rsplit("/", 1)[-1]
        archive = download(url, RAW / name)
        extracted = RAW / name.replace(".zip", "")
        if not extracted.exists():
            with zipfile.ZipFile(archive) as zf:
                zf.extractall(extracted)
        directories.append(extracted)
    return directories

if USE_OPENSLR:
    openslr_dirs = fetch_openslr()
    print("OpenSLR ready:", [d.name for d in openslr_dirs])

  fetching https://www.openslr.org/resources/86/yo_ng_female.zip
  462 MB
  fetching https://www.openslr.org/resources/86/yo_ng_male.zip
  445 MB
OpenSLR ready: ['yo_ng_female', 'yo_ng_male']


In [9]:
import os
import subprocess
from huggingface_hub import hf_hub_download


from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")

# REMEMBER: Use your new, secure token!
os.environ["HF_TOKEN"] = secret_value_0

In [10]:
# --- Hugging Face parquet sources --------------------------------------------
from huggingface_hub import HfApi, hf_hub_download

# Verified repository layouts (checked against the Hub API):
#
#   michsethowusu/yoruba-speech-text-parallel
#       data/train-00000-of-00050-*.parquet    50 shards, ~420 MB each, 21.5 GB
#
#   google/WaxalNLP
#       data/TTS/yor/{yor,yor_tts}-{train,validation,test}-*.parquet
#       11 files, 90-913 MB each, 2.6 GB
MICHSETHOWUSU_REPO, MICHSETHOWUSU_PATH = "michsethowusu/yoruba-speech-text-parallel", "data/"
WAXAL_REPO, WAXAL_PATH = "google/WaxalNLP", "data/TTS/yor/"


def fetch_hf_parquets(repo_id: str, path_prefix: str, limit: int | None = None,
                      splits: tuple[str, ...] = ("train",),
                      max_gb: float | None = None) -> list[Path]:
    """Download parquet shards under `path_prefix`.

    Downloads are deliberately *not* limited to the training budget. Fetching
    one shard and using all of it gives a sample drawn from one shard; fetching
    many and sampling across them gives the same hours of audio with far more
    speaker and session variety, which is what actually helps an acoustic model.
    Bandwidth on Kaggle is cheap; the scarce resource is training time.

    Args:
        repo_id: Exactly ``namespace/name``; deeper paths go in `path_prefix`.
        path_prefix: Directory within the repo, e.g. ``data/TTS/yor/``.
        limit: Optional cap on shard count. ``None`` takes everything matching.
        splits: Keep only files whose name contains one of these.
        max_gb: Optional disk guard. ``None`` means no limit.

    Raises:
        SystemExit: Listing the available paths, when nothing matches.
    """
    if repo_id.count("/") != 1:
        raise SystemExit(
            f"repo_id must be 'namespace/name', got {repo_id!r}. "
            "Put the directory in path_prefix instead."
        )

    info = HfApi().repo_info(repo_id, repo_type="dataset", files_metadata=True)
    candidates = [
        f for f in info.siblings
        if f.rfilename.endswith(".parquet")
        and f.rfilename.startswith(path_prefix)
        and any(s in Path(f.rfilename).name for s in splits)
    ]

    if not candidates:
        print(f"Nothing under {path_prefix!r} matching {splits} in {repo_id}.")
        print("Available parquet paths:")
        for f in sorted(x.rfilename for x in info.siblings if x.rfilename.endswith(".parquet"))[:25]:
            print(f"   {f}")
        raise SystemExit("Adjust path_prefix to one of the paths above.")

    candidates.sort(key=lambda f: f.rfilename)
    chosen, total_bytes = [], 0
    for shard in candidates:
        if limit is not None and len(chosen) >= limit:
            break
        size = shard.size or 0
        if max_gb is not None and (total_bytes + size) / 1e9 > max_gb:
            continue          # skip this one, a smaller later shard may still fit
        chosen.append(shard)
        total_bytes += size

    print(f"{repo_id}: {len(candidates)} files match {path_prefix} {splits}; "
          f"downloading {len(chosen)} ({total_bytes / 1e9:.2f} GB)")

    paths = []
    for index, shard in enumerate(chosen, start=1):
        paths.append(Path(hf_hub_download(
            repo_id, shard.rfilename, repo_type="dataset", cache_dir=str(RAW / "hf")
        )))
        if index % 5 == 0 or index == len(chosen):
            print(f"   {index}/{len(chosen)} shards")
    return paths


michsethowusu_files, waxal_files = [], []

if USE_MICHSETHOWUSU:
    michsethowusu_files = fetch_hf_parquets(
        MICHSETHOWUSU_REPO, MICHSETHOWUSU_PATH, MICHSETHOWUSU_SHARDS,
        splits=HF_SPLITS, max_gb=MAX_DOWNLOAD_GB.get("michsethowusu"),
    )

if USE_WAXAL:
    # data/TTS/yor holds two naming families (yor-* and yor_tts-*) across all
    # three splits; every one of them is usable audio here.
    waxal_files = fetch_hf_parquets(
        WAXAL_REPO, WAXAL_PATH, WAXAL_SHARDS,
        splits=HF_SPLITS, max_gb=MAX_DOWNLOAD_GB.get("waxal"),
    )


michsethowusu/yoruba-speech-text-parallel: 50 files match data/ ('train', 'validation', 'test'); downloading 50 (21.52 GB)


data/train-00000-of-00050-f8c3a0a96c9fec(…):   0%|          | 0.00/413M [00:00<?, ?B/s]

data/train-00001-of-00050-c15a2ffbc0c461(…):   0%|          | 0.00/428M [00:00<?, ?B/s]

data/train-00002-of-00050-b48ea36e7030bd(…):   0%|          | 0.00/450M [00:00<?, ?B/s]

data/train-00003-of-00050-b9cbf7ac7fd06b(…):   0%|          | 0.00/436M [00:00<?, ?B/s]

...

data/TTS/yor/yor_tts-train-00003.parquet:   0%|          | 0.00/244M [00:00<?, ?B/s]

data/TTS/yor/yor_tts-train-00004.parquet:   0%|          | 0.00/128M [00:00<?, ?B/s]

   10/11 shards


data/TTS/yor/yor_tts-validation-00000.pa(…):   0%|          | 0.00/103M [00:00<?, ?B/s]

   11/11 shards


---
## 3. Merge into one MFA corpus

Everything is normalized to the layout MFA expects: **16 kHz mono WAV** plus a
matching `.lab` transcript, grouped into per-speaker directories.

Speaker grouping matters. MFA's speaker-adaptive stage estimates a transform per
speaker, so utterances must be attributed correctly. Sources without speaker
labels get one synthetic speaker per source, which is the honest representation
of what is known.

Every utterance is filtered on duration and, when required, on diacritics.

In [11]:
import soundfile as sf
import numpy as np

class CorpusWriter:
    '''Writes utterances into an MFA-format corpus, with filtering and caps.'''

    def __init__(self, root: Path, min_s: float, max_s: float, require_tone: bool,
                 holdout_root: Path | None = None, holdout_speakers: set | None = None):
        self.root = root
        self.holdout_root = holdout_root
        # Bind the caller's set by reference, not `or set()`: an empty set is
        # falsy, so `or` would silently substitute a fresh one and every later
        # update to the caller's set would be invisible here.
        self.holdout_speakers = holdout_speakers if holdout_speakers is not None else set()
        self.holdout_hours = 0.0
        self.holdout_count = 0
        self.min_s = min_s
        self.max_s = max_s
        self.require_tone = require_tone
        self.hours = {}
        self.counts = {}
        self.rejected = {"short": 0, "long": 0, "no_tone": 0, "empty": 0, "unreadable": 0}
        self.index_rows = []

    def _resample_mono16k(self, samples, rate):
        'Downmix to mono and resample to 16 kHz, which is what MFA works in.'
        if getattr(samples, "ndim", 1) > 1:
            samples = samples.mean(axis=1)
        if rate != 16000:
            count = int(len(samples) / rate * 16000)
            samples = np.interp(
                np.linspace(0, len(samples) - 1, count),
                np.arange(len(samples)), samples,
            )
        return samples.astype("float32")

    def add(self, source: str, speaker: str, utt_id: str, text: str,
            audio_bytes: bytes | None = None, audio_path: Path | None = None) -> bool:
        'Add one utterance. Returns whether it was kept.'
        text = normalize(text or "")
        if not text.strip():
            self.rejected["empty"] += 1
            return False
        if self.require_tone and not has_tone_marks(text):
            self.rejected["no_tone"] += 1
            return False

        cap = MAX_HOURS.get(source)
        if cap is not None and self.hours.get(source, 0.0) >= cap:
            return False

        try:
            if audio_bytes is not None:
                samples, rate = sf.read(io.BytesIO(audio_bytes), dtype="float32", always_2d=False)
            else:
                samples, rate = sf.read(str(audio_path), dtype="float32", always_2d=False)
        except Exception:
            self.rejected["unreadable"] += 1
            return False

        duration = len(samples) / rate if rate else 0.0
        if duration < self.min_s:
            self.rejected["short"] += 1
            return False
        if duration > self.max_s:
            self.rejected["long"] += 1
            return False

        samples = self._resample_mono16k(samples, rate)

        # Held-out speakers go to a separate corpus and are never trained on.
        held_out = f"{source}_{speaker}" in self.holdout_speakers
        root = self.holdout_root if (held_out and self.holdout_root) else self.root
        speaker_dir = root / f"{source}_{speaker}"
        speaker_dir.mkdir(parents=True, exist_ok=True)

        sf.write(str(speaker_dir / f"{utt_id}.wav"), samples, 16000, subtype="PCM_16")
        (speaker_dir / f"{utt_id}.lab").write_text(text, encoding="utf-8")

        if held_out and self.holdout_root:
            self.holdout_hours += duration / 3600
            self.holdout_count += 1
        else:
            self.hours[source] = self.hours.get(source, 0.0) + duration / 3600
            self.counts[source] = self.counts.get(source, 0) + 1
            self.index_rows.append((utt_id, f"{source}_{speaker}", text))
        return True

    def summary(self):
        print(f"\n{'source':<18} {'utterances':>11} {'hours':>8}")
        for source in sorted(self.counts):
            print(f"{source:<18} {self.counts[source]:>11} {self.hours[source]:>8.2f}")
        print(f"{'TOTAL (train)':<18} {sum(self.counts.values()):>11} {sum(self.hours.values()):>8.2f}")
        if self.holdout_count:
            print(f"{'held out':<18} {self.holdout_count:>11} {self.holdout_hours:>8.2f}")
        print(f"\nrejected: {self.rejected}")

HOLDOUT = TEMP / "holdout_corpus"
HOLDOUT.mkdir(parents=True, exist_ok=True)

# Speakers are chosen before ingestion so the decision cannot depend on what was
# read first, which would bias the held-out set toward whatever appears early in
# the index.
holdout_speakers: set[str] = set()

writer = CorpusWriter(CORPUS, MIN_DURATION_S, MAX_DURATION_S, REQUIRE_TONE_MARKS,
                      holdout_root=HOLDOUT, holdout_speakers=holdout_speakers)
print("writer ready")

writer ready


In [12]:
# --- OpenSLR: flat WAV directory plus line_index.tsv -------------------------
if USE_OPENSLR and HOLDOUT_SPEAKER_FRACTION > 0:
    # Decide the held-out speakers up front, from the ids alone.
    import random as _random
    all_speakers = set()
    for directory in openslr_dirs:
        index = next(directory.rglob("line_index.tsv"), None)
        if index is None:
            continue
        with index.open(encoding="utf-8", newline="") as handle:
            for row in csv.reader(handle, delimiter="\t"):
                row = [c.strip() for c in row if c.strip()]
                if len(row) < 2:
                    continue
                head = row[0].rsplit("_", 1)
                if len(head) == 2 and head[1].isdigit():
                    all_speakers.add(f"openslr_{head[0]}")

    ordered = sorted(all_speakers)
    _random.Random(0).shuffle(ordered)
    count = max(1, int(len(ordered) * HOLDOUT_SPEAKER_FRACTION))
    holdout_speakers.update(ordered[:count])
    print(f"holding out {len(holdout_speakers)} of {len(ordered)} SLR86 speakers:")
    print("  " + ", ".join(sorted(holdout_speakers)))

    # The writer must see the same set object, or held-out speakers silently
    # end up in the training corpus.
    assert writer.holdout_speakers is holdout_speakers, (
        "writer is not sharing the holdout set; re-run the CorpusWriter cell"
    )

if USE_OPENSLR:
    for directory in openslr_dirs:
        index = next(directory.rglob("line_index.tsv"), None)
        if index is None:
            print(f"  no line_index.tsv in {directory}")
            continue
        base = index.parent
        kept = 0
        with index.open(encoding="utf-8", newline="") as handle:
            for row in csv.reader(handle, delimiter="\t"):
                row = [c.strip() for c in row if c.strip()]
                if len(row) < 2:
                    continue
                utt_id, text = row[0], row[-1]
                wav = base / f"{utt_id}.wav"
                if not wav.exists():
                    continue
                # yof_06136_00616155826 -> speaker yof_06136
                head = utt_id.rsplit("_", 1)
                speaker = head[0] if len(head) == 2 and head[1].isdigit() else directory.name
                kept += writer.add("openslr", speaker, utt_id, text, audio_path=wav)
        print(f"  {directory.name}: kept {kept}")
    writer.summary()

holding out 5 of 36 SLR86 speakers:
  openslr_yof_00295, openslr_yof_07508, openslr_yom_00295, openslr_yom_01208, openslr_yom_01523
  yo_ng_female: kept 1892
  yo_ng_male: kept 1691

source              utterances    hours
openslr                   3083     3.47
TOTAL (train)             3083     3.47
held out                   500     0.56

rejected: {'short': 0, 'long': 0, 'no_tone': 0, 'empty': 0, 'unreadable': 0}


In [13]:
# --- michsethowusu: parquet with audio struct + text -------------------------
import pyarrow.parquet as pq

def ingest_parquet(paths, source, speaker_field=None, id_prefix=""):
    'Stream parquet shards into the corpus, batch by batch to bound memory.'
    total = 0
    unlabelled = 0   # counts kept clips with no speaker label, for bucketing
    for path in paths:
        parquet = pq.ParquetFile(path)
        for batch_index, batch in enumerate(parquet.iter_batches(batch_size=500)):
            data = batch.to_pydict()
            texts = data.get("text") or data.get("sentence") or []
            audios = data.get("audio") or []
            speakers = data.get(speaker_field) if speaker_field else None

            for row in range(len(texts)):
                audio = audios[row]
                if not audio or not audio.get("bytes"):
                    continue
                label = (speakers[row] if speakers else None) or None
                if label is None:
                    # No speaker field: bucket into pseudo-speakers rather than
                    # dropping every clip into one directory. "_unk" in the name
                    # marks the group as not a real speaker; later stages test
                    # for it before doing anything per-speaker.
                    label = f"unk{unlabelled // BASE_BUCKET_SIZE:05d}"
                utt_id = f"{id_prefix}{path.stem}_{batch_index}_{row}"
                kept_one = writer.add(source, str(label), utt_id, texts[row],
                                      audio_bytes=audio["bytes"])
                total += kept_one
                if kept_one and speakers is None:
                    unlabelled += 1
            if writer.hours.get(source, 0) >= MAX_HOURS.get(source, 1e9):
                print(f"  {source}: hit the {MAX_HOURS[source]} h cap")
                return total
        print(f"  {path.name}: running total {total}")
    return total

if USE_MICHSETHOWUSU:
    ingest_parquet(michsethowusu_files, "michsethowusu", id_prefix="mh_")
    writer.summary()

  train-00000-of-00050-f8c3a0a96c9fec73.parquet: running total 14266
  train-00001-of-00050-c15a2ffbc0c4618f.parquet: running total 28864
  train-00002-of-00050-b48ea36e7030bd37.parquet: running total 44396
  train-00003-of-00050-b9cbf7ac7fd06b46.parquet: running total 59052
  train-00004-of-00050-d44569683a7dd02e.parquet: running total 73976
  train-00005-of-00050-c5c0bde5b53a4071.parquet: running total 88955
  michsethowusu: hit the 24.0 h cap

source              utterances    hours
michsethowusu            99550    24.00
openslr                   3083     3.47
TOTAL (train)           102633    27.47
held out                   500     0.56

rejected: {'short': 121732, 'long': 0, 'no_tone': 0, 'empty': 0, 'unreadable': 0}


In [14]:
# --- WaxalNLP yor_tts: long MP3 clips, speaker labelled ----------------------
if USE_WAXAL:
    ingest_parquet(waxal_files, "waxal", speaker_field="speaker_id", id_prefix="wx_")
    writer.summary()

  yor-test-00000.parquet: running total 75
  yor-train-00000.parquet: running total 864
  yor-train-00001.parquet: running total 1016
  yor-validation-00000.parquet: running total 1101
  yor_tts-test-00000.parquet: running total 1176
  yor_tts-train-00000.parquet: running total 1430
  yor_tts-train-00001.parquet: running total 1661
  yor_tts-train-00002.parquet: running total 1847
  yor_tts-train-00003.parquet: running total 2032
  yor_tts-train-00004.parquet: running total 2117
  yor_tts-validation-00000.parquet: running total 2202

source              utterances    hours
michsethowusu            99550    24.00
openslr                   3083     3.47
waxal                     2202     6.30
TOTAL (train)           104835    33.77
held out                   500     0.56

rejected: {'short': 121732, 'long': 3020, 'no_tone': 0, 'empty': 0, 'unreadable': 0}


In [15]:
# --- Your Yoruba-TTS-Dataset, attached as a Kaggle dataset -------------------
if USE_LOCAL_TTS:
    if not LOCAL_TTS_DIR.exists():
        raise SystemExit(
            f"{LOCAL_TTS_DIR} not found. Attach the dataset via 'Add Input', then "
            "set LOCAL_TTS_DIR to the path shown under /kaggle/input."
        )

    sessions = [d for d in sorted(LOCAL_TTS_DIR.rglob("mapping.tsv"))]
    print(f"found {len(sessions)} session mappings")
    kept = 0
    for mapping in sessions:
        session = mapping.parent
        with mapping.open(encoding="utf-8", newline="") as handle:
            for row in csv.DictReader(handle, delimiter="\t"):
                filename = (row.get("audio_filename") or "").strip()
                text = (row.get("sentence") or "").strip()
                if not filename or not text or filename.startswith("._"):
                    continue
                clip = session / filename
                if not clip.exists():
                    continue
                utt_id = f"tts_{session.name[-13:]}_{Path(filename).stem}"
                kept += writer.add("local_tts", session.name, utt_id, text, audio_path=clip)
    print(f"  kept {kept}")
    writer.summary()

found 17 session mappings
  kept 1606

source              utterances    hours
local_tts                 1606     4.16
michsethowusu            99550    24.00
openslr                   3083     3.47
waxal                     2202     6.30
TOTAL (train)           106441    37.93
held out                   500     0.56

rejected: {'short': 121732, 'long': 3332, 'no_tone': 0, 'empty': 0, 'unreadable': 0}


---
## 4. Build the pronunciation dictionary

Rule-based, so this takes seconds regardless of corpus size.

### Tone-suffixed or tone-stripped phones?

Tone-suffixed phones (`a_H`/`a_M`/`a_L`) let the model learn tone-conditioned
vowels and give sharper boundaries, at the cost of tripling the vowel inventory.

**Any undiacritized source forces tone-stripped phones.** Text without
diacritics reads as all-Mid, so including it in a tone-suffixed dictionary
teaches the model that Mid vowels sound like every other tone — actively worse
than not having the data.

In [16]:
import re

undiacritized_sources = (USE_WAXAL or USE_LOCAL_TTS)
TONE_IN_PHONES = (not undiacritized_sources) and sum(writer.hours.values()) >= 5.0

print(f"undiacritized sources enabled: {undiacritized_sources}")
print(f"corpus hours: {sum(writer.hours.values()):.2f}")
print(f"=> TONE_IN_PHONES = {TONE_IN_PHONES}")

vocabulary = sorted({
    token for _, _, text in writer.index_rows for token in tokenize(text, keep_noise=False)
})
print(f"vocabulary: {len(vocabulary)} words")

dictionary = PronunciationDictionary()
dictionary.build(vocabulary)

if not TONE_IN_PHONES:
    stripped = {}
    for word, variants in dictionary.entries.items():
        seen = []
        for variant in variants:
            phones = " ".join(re.sub(r"_[HML]$", "", p) for p in variant.split())
            if phones not in seen:
                seen.append(phones)
        stripped[word] = seen
    dictionary.entries = stripped

DICT_PATH = OUTPUT / "yoruba.dict"
dictionary.write(DICT_PATH)
print(f"entries {len(dictionary.entries)}, phones {len(dictionary.phone_inventory())}, "
      f"failures {len(dictionary.failures)}")
if dictionary.failures:
    print("first failures:", list(dictionary.failures)[:10])

undiacritized sources enabled: True
corpus hours: 37.93
=> TONE_IN_PHONES = False
vocabulary: 9266 words
entries 8963, phones 31, failures 304
first failures: ['$1', '-', '--', '0.2', '0.3', '1', '1.5', '10', '100', '1000']


In [17]:
# Words the rules cannot convert are dropped from the corpus rather than left to
# fail the whole alignment: MFA treats them as out-of-vocabulary and aborts the
# utterance containing them.
bad = set(dictionary.failures)
if bad:
    removed = 0
    for utt_id, speaker, text in writer.index_rows:
        if any(token in bad for token in tokenize(text, keep_noise=False)):
            for suffix in (".wav", ".lab"):
                path = CORPUS / speaker / f"{utt_id}{suffix}"
                if path.exists():
                    path.unlink()
            removed += 1
    print(f"removed {removed} utterances containing unmappable words")
else:
    print("no unmappable words")

wavs = list(CORPUS.rglob("*.wav"))
print(f"corpus: {len(wavs)} utterances across "
      f"{len([d for d in CORPUS.iterdir() if d.is_dir()])} speakers")

removed 1113 utterances containing unmappable words
corpus: 105328 utterances across 2047 speakers


---
### Self-check

Verifies the failures that are silent: an empty held-out set, a source that
contributed nothing, dictionary gaps, a tone setting inconsistent with the
text. Cheap to run, and catches problems before a multi-hour train does.


In [18]:
# --- Pipeline self-check ------------------------------------------------------
# Run after the corpus is built and before training. Verifies the things that
# fail silently: an empty held-out set, a source that contributed nothing, a
# dictionary that does not cover the corpus.
problems, notes = [], []

train_wavs = list(CORPUS.rglob("*.wav"))
train_labs = list(CORPUS.rglob("*.lab"))
hold_wavs = list(HOLDOUT.rglob("*.wav"))
train_speakers = [d for d in CORPUS.iterdir() if d.is_dir()]
hold_speakers = [d for d in HOLDOUT.iterdir() if d.is_dir()]

print(f"{'training utterances':<26}: {len(train_wavs)}")
print(f"{'training transcripts':<26}: {len(train_labs)}")
print(f"{'training speakers':<26}: {len(train_speakers)}")
print(f"{'training hours':<26}: {sum(writer.hours.values()):.2f}")
print(f"{'held-out utterances':<26}: {len(hold_wavs)}")
print(f"{'held-out speakers':<26}: {len(hold_speakers)}")
print(f"{'held-out hours':<26}: {writer.holdout_hours:.2f}")

if len(train_wavs) != len(train_labs):
    problems.append(f"{len(train_wavs)} audio files but {len(train_labs)} transcripts; "
                    "MFA needs one .lab per .wav")

if HOLDOUT_SPEAKER_FRACTION > 0 and USE_OPENSLR and not hold_wavs:
    problems.append(
        "HOLDOUT_SPEAKER_FRACTION > 0 but nothing was held out. The writer and "
        "the speaker-selection cell must share one set object -- re-run the "
        "CorpusWriter cell, then the OpenSLR cell, in that order."
    )

for name, enabled in [("openslr", USE_OPENSLR), ("michsethowusu", USE_MICHSETHOWUSU),
                      ("waxal", USE_WAXAL), ("local_tts", USE_LOCAL_TTS)]:
    if enabled and not writer.counts.get(name):
        problems.append(f"{name} is enabled but contributed no utterances")
    elif enabled:
        share = writer.hours[name] / max(sum(writer.hours.values()), 1e-9)
        notes.append(f"{name}: {writer.counts[name]} utterances, "
                     f"{writer.hours[name]:.2f} h ({share * 100:.0f}% of training audio)")

# Every word in the corpus must be in the dictionary, or MFA drops the utterance.
corpus_words = {w for _, _, text in writer.index_rows
                for w in tokenize(text, keep_noise=False)}
missing = corpus_words - set(dictionary.entries)
if missing:
    problems.append(f"{len(missing)} corpus words are missing from the dictionary, "
                    f"e.g. {sorted(missing)[:5]}")

toned = sum(1 for _, _, text in writer.index_rows if has_tone_marks(text))
coverage = toned / max(len(writer.index_rows), 1)
notes.append(f"tone-mark coverage across training text: {coverage * 100:.1f}%")
if TONE_IN_PHONES and coverage < 0.95:
    problems.append(f"TONE_IN_PHONES is on but only {coverage:.0%} of lines carry "
                    "tone marks; undiacritized text would be labelled all-Mid")

print("\n".join(["", "Notes:"] + [f"  {n}" for n in notes]))
if problems:
    print("\nProblems:")
    for problem in problems:
        print(f"  ! {problem}")
else:
    print("\nNo problems found. Ready to train.")


training utterances       : 105328
training transcripts      : 105328
training speakers         : 2047
training hours            : 37.93
held-out utterances       : 500
held-out speakers         : 5
held-out hours            : 0.56

Notes:
  openslr: 3083 utterances, 3.47 h (9% of training audio)
  michsethowusu: 99550 utterances, 24.00 h (63% of training audio)
  waxal: 2202 utterances, 6.30 h (17% of training audio)
  local_tts: 1606 utterances, 4.16 h (11% of training audio)
  tone-mark coverage across training text: 88.9%

Problems:
  ! 304 corpus words are missing from the dictionary, e.g. ['$1', '-', '--', '0.2', '0.3']


---
## 5. Validate

Catches out-of-vocabulary words and unreadable audio before a multi-hour train
discovers them the expensive way.

In [19]:
import time

def run_mfa(args, description=""):
    'Run an MFA subcommand, hiding progress-bar redraws.'
    command = [str(MFA_BIN), *[str(a) for a in args]]
    # Postgres avoids the SQLite lock contention that killed a 5.5-hour run.
    if globals().get("USE_POSTGRES") and args and args[0] in ("train", "align", "validate"):
        command.append("--use_postgres")
    print(f"=== {description} ===\n{' '.join(command)}\n", flush=True)

    env = {**os.environ}
    env["PATH"] = f"{MFA_ENV / 'bin'}{os.pathsep}{env['PATH']}"
    env["TERM"] = "dumb"

    progress = re.compile(r"[━╸]|\d+%\s|it/s")
    started = last = time.time()
    hidden = 0

    process = subprocess.Popen(command, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
    for line in process.stdout:
        if progress.search(line):
            hidden += 1
            if time.time() - last >= 60:
                print(f"  ... {(time.time() - started) / 60:.0f} min elapsed", flush=True)
                last = time.time()
            continue
        if line.strip():
            print(line, end="", flush=True)
    process.wait()

    print(f"\nfinished in {(time.time() - started) / 60:.1f} min "
          f"(exit {process.returncode}, {hidden} progress lines hidden)")
    if process.returncode != 0:
        raise RuntimeError(f"MFA exited {process.returncode}")

run_mfa(["validate", CORPUS, DICT_PATH, "--num_jobs", NUM_JOBS, "--clean"],
        "Corpus validation")

=== Corpus validation ===
/kaggle/temp/mfa-env/bin/mfa validate /kaggle/temp/mfa_corpus /kaggle/working/yoruba_model/yoruba.dict --num_jobs 4 --clean

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
  ... 1 min elapsed
 INFO     Found 2047 speakers across 105328 files, average number of utterances 
          per speaker: 51.45481191988276                                        
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 INFO     Generating MFCCs...                                                   
  ... 4 min elapsed
 INFO     Calculating CMVN...                                                   
 INFO     Generating final features...                                          
 INFO     Creating corpus split...                                              

---
## 6. Train

The long step. Watch the clock against Kaggle's 12-hour session limit — if the
corpus is much over 5 hours of audio this will not finish, and an interrupted
run leaves no model.

If you are close to the limit, train monophone-only first: it is far quicker and
still produces a usable, if blunter, aligner.

In [20]:
ACOUSTIC_MODEL = OUTPUT / "yoruba_acoustic.zip"
ALIGNED = TEMP / "aligned"
ALIGNED.mkdir(exist_ok=True)

# Reuse an acoustic model instead of training a new one.
#
# Training is by far the most expensive step (4-5 h); aligning the same corpus
# with a model you already trust takes well under an hour, and Part 2 only
# needs the alignments. Set this when the aligner is good and you are iterating
# on the vowel classifier rather than on the model.
#
# Attach the model as a Kaggle dataset and point at the directory holding
# yoruba_acoustic.zip. The dictionary is NOT taken from there -- this run
# rebuilds it from its own corpus vocabulary, so every word is covered -- but
# the phone sets must still agree, which is checked below.
PRETRAINED = None
# PRETRAINED = Path("/kaggle/input/datasets/pearsejim01/yoruba-model")

total_hours = sum(writer.hours.values())
print(f"about to train on {total_hours:.2f} hours with {NUM_JOBS} jobs")
print(f"rough estimate: {total_hours * KAGGLE_HOURS_PER_AUDIO_HOUR * (KAGGLE_CORES / NUM_JOBS):.1f} h "
      f"(measured reference: 18 h of audio finished in 2.5 h on Kaggle)")
print("Runtime tracks utterance count and sample rate more than hours of audio, "
      "so watch the elapsed-time output rather than the estimate.")

about to train on 37.93 hours with 4 jobs
rough estimate: 5.3 h (measured reference: 18 h of audio finished in 2.5 h on Kaggle)
Runtime tracks utterance count and sample rate more than hours of audio, so watch the elapsed-time output rather than the estimate.


In [21]:
if PRETRAINED:
    import shutil, zipfile

    uploaded = PRETRAINED / "yoruba_acoustic.zip"
    if not uploaded.exists():
        raise SystemExit(f"{uploaded} not found; check the attached dataset path")

    # A model trained on tone-stripped phones cannot align against a
    # tone-suffixed dictionary. MFA reports no error when they disagree -- it
    # silently emits nonsense -- so refuse to start on a mismatch.
    with zipfile.ZipFile(uploaded) as archive:
        name = next(n for n in archive.namelist() if n.endswith("phones.txt"))
        model_phones = {line.split()[0] for line in
                        archive.read(name).decode("utf-8").splitlines() if line.split()}
    missing = dictionary.phone_inventory() - model_phones
    if missing:
        raise SystemExit(
            f"{len(missing)} phones in this run's dictionary are absent from the "
            f"uploaded model, e.g. {sorted(missing)[:8]}. The model and the "
            "dictionary use different phone styles; retrain or rebuild to match."
        )
    print(f"phone sets agree ({len(dictionary.phone_inventory())} phones)")

    shutil.copy2(uploaded, ACOUSTIC_MODEL)
    run_mfa(
        ["align", CORPUS, DICT_PATH, ACOUSTIC_MODEL, ALIGNED,
         "--num_jobs", NUM_JOBS, "--clean"],
        "Aligning with the uploaded model (training skipped)",
    )
else:
    run_mfa(
        ["train", CORPUS, DICT_PATH, ACOUSTIC_MODEL,
         "--num_jobs", NUM_JOBS, "--clean",
         "--output_directory", ALIGNED],
        "Acoustic model training",
    )

=== Acoustic model training ===
/kaggle/temp/mfa-env/bin/mfa train /kaggle/temp/mfa_corpus /kaggle/working/yoruba_model/yoruba.dict /kaggle/working/yoruba_model/yoruba_acoustic.zip --num_jobs 4 --clean --output_directory /kaggle/temp/aligned

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
  ... 1 min elapsed
 INFO     Found 2047 speakers across 105328 files, average number of utterances 
          per speaker: 51.45481191988276                                        
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 INFO     Generating MFCCs...                                                   
  ... 4 min elapsed
 INFO     Calculating CMVN...                                                   
 INFO     Generating final features...                               

---
## 7. Evaluate on held-out speakers

Training always "succeeds" -- MFA produces a model regardless. The question is
whether it generalises to a voice it has never heard, which is what every learner
is.

The speakers withheld in section 3 were never trained on. Aligning them now gives
an out-of-sample measure of alignment success, phone-duration sanity, and tone
recovery. SLR86 is used because it is the only fully diacritized source, so its
written tones are free ground truth.

Compare against the current production baseline before adopting a new model:
**99.5% alignment success, 107 ms mean vowel duration, 74.7% tone accuracy.**


In [22]:
held_out_wavs = list(HOLDOUT.rglob("*.wav"))
print(f"held-out corpus: {len(held_out_wavs)} utterances, "
      f"{writer.holdout_hours:.2f} h, "
      f"{len([d for d in HOLDOUT.iterdir() if d.is_dir()])} speakers")

if not held_out_wavs:
    print("\nNothing held out -- set HOLDOUT_SPEAKER_FRACTION above 0 to enable "
          "out-of-sample evaluation.")

held-out corpus: 500 utterances, 0.56 h, 5 speakers


In [23]:
HOLDOUT_ALIGNED = TEMP / "holdout_aligned"
HOLDOUT_ALIGNED.mkdir(exist_ok=True)

if held_out_wavs:
    run_mfa(
        ["align", HOLDOUT, DICT_PATH, ACOUSTIC_MODEL, HOLDOUT_ALIGNED,
         "--num_jobs", NUM_JOBS, "--clean"],
        "Aligning held-out speakers",
    )

=== Aligning held-out speakers ===
/kaggle/temp/mfa-env/bin/mfa align /kaggle/temp/holdout_corpus /kaggle/working/yoruba_model/yoruba.dict /kaggle/working/yoruba_model/yoruba_acoustic.zip /kaggle/temp/holdout_aligned --num_jobs 4 --clean

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
 INFO     Found 5 speakers across 500 files, average number of utterances per   
          speaker: 100.0                                                        
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 INFO     Generating MFCCs...                                                   
 INFO     Calculating CMVN...                                                   
 INFO     Generating final features...                                          
 INFO     Creating corpus split.

In [24]:
import statistics
from collections import defaultdict

sys.path.insert(0, str(staged_root))
from yoruba_engine.alignment.textgrid import alignment_from_textgrid
from yoruba_engine.datatypes import Tone
from yoruba_engine.g2p.orthography import is_noise_token, tone_pattern
from yoruba_engine.pitch.extractor import is_vowel_phone

holdout_results = {}

if held_out_wavs:
    textgrids_out = list(HOLDOUT_ALIGNED.rglob("*.TextGrid"))
    attempted_out = len(held_out_wavs)
    success = len(textgrids_out) / attempted_out if attempted_out else 0.0

    durations = defaultdict(list)
    for textgrid in textgrids_out:
        try:
            alignment = alignment_from_textgrid(textgrid)
        except Exception:
            continue
        for phone in alignment.phones:
            durations[phone.label].append(phone.duration_s)

    vowel_durations = [d for label, values in durations.items()
                       if is_vowel_phone(label) for d in values]
    mean_vowel_ms = statistics.mean(vowel_durations) * 1000 if vowel_durations else 0.0

    holdout_results = {
        "utterances": attempted_out,
        "aligned": len(textgrids_out),
        "alignment_success": round(success, 4),
        "mean_vowel_ms": round(mean_vowel_ms, 1),
        "distinct_phones": len(durations),
    }

    print(f"alignment success : {success * 100:.1f}%   (baseline 99.5%)")
    print(f"mean vowel        : {mean_vowel_ms:.0f} ms   (baseline 107 ms, expected 60-150)")
    print(f"distinct phones   : {len(durations)}")

    if success < 0.95:
        print("\nBelow baseline. Usually too little data per phone, or a corpus "
              "dominated by one source. Check the per-source hours above.")
    if not 60 <= mean_vowel_ms <= 150:
        print("\nVowel durations are outside the expected range: boundaries are "
              "unreliable even where alignment reported success.")

alignment success : 100.0%   (baseline 99.5%)
mean vowel        : 97 ms   (baseline 107 ms, expected 60-150)
distinct phones   : 30


In [25]:
# Tone recovery on held-out speakers. Needs Parselmouth for pitch; if it is not
# present this is skipped rather than failing the run.
try:
    import parselmouth  # noqa: F401
    from yoruba_engine.config import EngineConfig
    from yoruba_engine.pitch.extractor import PitchExtractor
    from yoruba_engine.tone.analyzer import ToneAnalyzer
    have_pitch = True
except ImportError:
    print("Parselmouth unavailable; skipping tone evaluation.")
    print("  pip install praat-parselmouth")
    have_pitch = False

if held_out_wavs and have_pitch:
    engine_config = EngineConfig()
    extractor = PitchExtractor(engine_config.pitch)
    analyzer = ToneAnalyzer(engine_config.tone)

    audio_by_stem = {w.stem: w for w in held_out_wavs}
    confusion = defaultdict(int)
    checked = 0

    for textgrid in list(HOLDOUT_ALIGNED.rglob("*.TextGrid"))[:400]:
        audio = audio_by_stem.get(textgrid.stem)
        if audio is None:
            continue
        try:
            alignment = alignment_from_textgrid(textgrid)
            words, _ = extractor.extract(audio, alignment)
            words = analyzer.analyse(words)
        except Exception:
            continue
        checked += 1
        for word in words:
            if is_noise_token(word.word):
                continue
            expected, detected = tone_pattern(word.word), word.tone_pattern
            if len(expected) != len(detected):
                continue
            for exp, det in zip(expected, detected):
                if det is not Tone.UNKNOWN:
                    confusion[(exp, det)] += 1

    total = sum(confusion.values())
    correct = sum(v for (e, d), v in confusion.items() if e == d)
    if total:
        accuracy = correct / total
        holdout_results["tone_accuracy"] = round(accuracy, 4)
        print(f"\ntone accuracy on {checked} held-out utterances: "
              f"{correct}/{total} = {accuracy * 100:.1f}%")
        print("  (rule-based thresholds; the trained classifier scores higher, "
              "but must be retrained locally against this model)")

        names = {Tone.HIGH: "High", Tone.MID: "Mid", Tone.LOW: "Low"}
        print(f"\n{'expected \\ detected':<22}" + "".join(f"{n:>8}" for n in names.values()))
        for exp in names:
            row_total = sum(confusion[(exp, d)] for d in names)
            cells = "".join(f"{confusion[(exp, d)]:>8}" for d in names)
            recall = f"{confusion[(exp, exp)] / row_total * 100:>9.1f}%" if row_total else ""
            print(f"{names[exp]:<22}{cells}{recall}")
    else:
        print("\nNo tones measured on the held-out set.")


tone accuracy on 400 held-out utterances: 2655/4204 = 63.2%
  (rule-based thresholds; the trained classifier scores higher, but must be retrained locally against this model)

expected \ detected       High     Mid     Low
High                      1145     320     118     72.3%
Mid                        346     965     219     63.1%
Low                        187     359     545     50.0%


---
## 8. Export

`/kaggle/working` is what you download at the end of the session. The model and
its dictionary are a **matched pair** — a model trained on tone-stripped phones
cannot align against a tone-suffixed dictionary — so both are saved together
along with a manifest describing what produced them.

In [26]:
import json
from datetime import datetime, timezone

textgrids = list(ALIGNED.rglob("*.TextGrid"))
attempted = len(list(CORPUS.rglob("*.wav")))

manifest = {
    "created": datetime.now(timezone.utc).isoformat(),
    "trained_on": "kaggle",
    "sources": {s: {"utterances": writer.counts[s], "hours": round(writer.hours[s], 3)}
                for s in writer.counts},
    "total_hours": round(total_hours, 3),
    "speakers": len([d for d in CORPUS.iterdir() if d.is_dir()]),
    "dictionary": {
        "entries": len(dictionary.entries),
        "phones": len(dictionary.phone_inventory()),
        "tone_in_phones": TONE_IN_PHONES,
    },
    "filters": {
        "min_duration_s": MIN_DURATION_S,
        "max_duration_s": MAX_DURATION_S,
        "require_tone_marks": REQUIRE_TONE_MARKS,
    },
    "alignment": {
        "attempted": attempted,
        "textgrids": len(textgrids),
        "failure_rate": round(1 - len(textgrids) / attempted, 4) if attempted else None,
    },
    # Only record held-out speakers if they were genuinely excluded. Listing
    # them when the holdout corpus is empty would document a guarantee the model
    # does not have, and any later evaluation on them would be in-sample.
    "holdout_evaluation": holdout_results,
    "holdout_speakers": sorted(holdout_speakers) if writer.holdout_count else [],
    "holdout_applied": bool(writer.holdout_count),
}
(OUTPUT / "yoruba_acoustic.manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

{
  "created": "2026-08-15T20:45:55.461663+00:00",
  "trained_on": "kaggle",
  "sources": {
    "openslr": {
      "utterances": 3083,
      "hours": 3.467
    },
    "michsethowusu": {
      "utterances": 99550,
      "hours": 24.0
    },
    "waxal": {
      "utterances": 2202,
      "hours": 6.302
    },
    "local_tts": {
      "utterances": 1606,
      "hours": 4.16
    }
  },
  "total_hours": 37.93,
  "speakers": 2047,
  "dictionary": {
    "entries": 8963,
    "phones": 31,
    "tone_in_phones": false
  },
  "filters": {
    "min_duration_s": 0.3,
    "max_duration_s": 15.0,
    "require_tone_marks": false
  },
  "alignment": {
    "attempted": 105328,
    "textgrids": 104961,
    "failure_rate": 0.0035
  },
  "holdout_evaluation": {
    "utterances": 500,
    "aligned": 500,
    "alignment_success": 1.0,
    "mean_vowel_ms": 97.3,
    "distinct_phones": 30,
    "tone_accuracy": 0.6315
  },
  "holdout_speakers": [
    "openslr_yof_00295",
    "openslr_yof_07508",
    "openslr_yo

In [27]:
# Keep a sample of alignments so the tone classifier can be trained locally
# without re-running the aligner. All of them would be far too large.
import shutil, random

sample_dir = OUTPUT / "aligned_sample"
sample_dir.mkdir(exist_ok=True)
random.seed(0)
for textgrid in random.sample(textgrids, min(2000, len(textgrids))):
    shutil.copy2(textgrid, sample_dir / textgrid.name)

print(f"exported {len(list(sample_dir.iterdir()))} TextGrids")
print("\nFiles to download from /kaggle/working/yoruba_model:")
for path in sorted(OUTPUT.iterdir()):
    size = sum(f.stat().st_size for f in path.rglob('*')) if path.is_dir() else path.stat().st_size
    print(f"  {path.name:<32} {size / 1e6:>8.1f} MB")

exported 2000 TextGrids

Files to download from /kaggle/working/yoruba_model:
  aligned_sample                        3.5 MB
  yoruba.dict                           0.3 MB
  yoruba_acoustic.manifest.json         0.0 MB
  yoruba_acoustic.zip                  61.6 MB


---
## Using the result locally

1. Download `yoruba_model/` from the session output.
2. Copy `yoruba_acoustic.zip` and `yoruba.dict` into the project's `models/` and
   `data/dictionary/`.
3. Regenerate the derived statistics, which describe *this* model's behaviour
   and are wrong for any other:

```bash
python scripts/measure_alignment_stats.py
python scripts/train_tone_classifier.py
```

4. Re-run section 9 of `train_mfa_yoruba.ipynb` to check tone accuracy did not
   regress. More data does not automatically mean better alignment: adding a
   large volume of isolated single words can make a model worse at connected
   speech, which is what learners actually produce.

**Compare against the current baseline before adopting it**: 99.5% alignment
success, 74.7% tone detection on held-out speakers. If the new model does not
beat that, keep the old one.

## Troubleshooting

**`ThirdpartyError: Could not find 'fstcompile'`**
`MFA_ENV/bin` is missing from `PATH`; re-run the install cell.

**Session dies during training**
Too much audio. Lower `MAX_HOURS`, restart, re-run.

**Out of disk**
Raw downloads go to `/kaggle/temp`, but the converted corpus does too. Reduce
the shard counts.

**Everything is out of vocabulary**
The dictionary was built from a different vocabulary than the corpus, usually
because a cell was re-run out of order. Re-run sections 3 and 4 in sequence.

---
# Part 2 — Nasal mining and the vowel classifier

Everything above trains the **acoustic model**, which decides *where* each sound
is. This part trains the **vowel classifier**, which decides *which* vowel was
actually produced -- the check that catches a learner saying `òkàn` (heart) when
the prompt was `òkun` (ocean). Alignment cannot see that error: the wrong word
has the right shape and occupies plausible time.

## Why mining, rather than just more data

The classifier's weakness is entirely in the five nasal vowels, and it is caused
by imbalance rather than by the features. Measured F1 tracks the oral:nasal ratio
almost exactly:

| pair | ratio | F1 |
|---|---|---|
| `u`/`un` | 1.6:1 | 0.45 |
| `ọ`/`ọn` | 4.6:1 | 0.29 |
| `a`/`an` | 10.4:1 | 0.14 |
| `i`/`in` | 17.9:1 | 0.01 |
| `ẹ`/`ẹn` | 58.5:1 | 0.00 |

Sampling more data at random does not help: it multiplies both classes equally
and leaves the ratio untouched. Class weighting helps only where there are enough
examples to weight -- it lifted `an` (1,108 samples) substantially and `ẹn`
(73 samples) barely at all, because you cannot re-weight your way out of 73
examples.

**Targeted mining changes the ratio.** Roughly 25% of michsethowusu clips contain
a nasal vowel, so selecting only those yields about 10,400 nasal vowels per
shard -- more than twice the entire base corpus.

A local run over 2,400 mined clips produced:

| | before | after |
|---|---|---|
| `ọn` F1 | 0.286 | **0.483** |
| `in` F1 | 0.195 | **0.297** |
| macro F1 | 0.493 | **0.525** |
| `òkun` vs `òkan` | ranked **wrong** way | ranked **correctly** |

The discrimination that heart-vs-ocean depends on started working. It was still
short of the confidence needed to report a verdict to a learner, which is what
scaling this up on Kaggle is for.

## Configuration

In [28]:
# Where the acoustic model comes from. Part 1 writes one; set this to an
# uploaded model instead if you would rather keep an existing aligner -- more
# audio does not automatically make a better one, so the model you already
# trust is often the right choice.
USE_MODEL = ACOUSTIC_MODEL if "ACOUSTIC_MODEL" in dir() else None
USE_DICT = DICT_PATH if "DICT_PATH" in dir() else None

# Or point at an uploaded pair. They must match: a model trained on
# tone-stripped phones cannot align against a tone-suffixed dictionary, and MFA
# reports no error when they disagree -- it silently emits nonsense.
# USE_MODEL = Path("/kaggle/input/datasets/pearsejim01/yoruba-model/yoruba_acoustic.zip")
# USE_DICT  = Path("/kaggle/input/datasets/pearsejim01/yoruba-model/yoruba.dict")

# --- How many nasal-bearing clips to mine -------------------------------------
# A fixed target dilutes as MAX_HOURS grows. The 20,000 used previously gave a
# 17.0% nasal share against a ~4 h base corpus and only 12.8% against a 48 h
# one, because the base corpus grew 13x while the mined pool did not. Scale it
# against the corpus actually built instead of hard-coding a number.
#
# Rates calibrated on the 48 h run (123,150 utterances -> 607,759 vowels,
# 77,536 of them nasal, including ~18,000 mined clips):
#
#   base corpus : 4.9 vowels per utterance, 10.3% of them nasal
#   mined clip  : 4.5 vowels per clip, 1.3 of them nasal (28.9%)
#
# Solving for the clip count k that hits a target nasal share s:
#
#   (base_nasal + 1.3k) / (base_vowels + 4.5k) = s
#   k = base_vowels * (s - 0.103) / (1.3 - 4.5s)
NASAL_SHARE_TARGET = 0.17
BASE_VOWELS_PER_UTT, BASE_NASAL_RATE = 4.9, 0.103
MINED_VOWELS_PER_CLIP, MINED_NASALS_PER_CLIP = 4.5, 1.3

# Alignment is the bottleneck, not mining: measured 90 s per 600 clips. Budget
# by wall-clock so the mining stage cannot eat the session on its own.
NASAL_ALIGN_BUDGET_H = 2.5
ALIGN_SECONDS_PER_CLIP = 90 / 600

base_utterances = sum(writer.counts.values())
base_vowels = BASE_VOWELS_PER_UTT * base_utterances
denominator = MINED_NASALS_PER_CLIP - NASAL_SHARE_TARGET * MINED_VOWELS_PER_CLIP
wanted = int(max(0.0, base_vowels * (NASAL_SHARE_TARGET - BASE_NASAL_RATE) / denominator))

budget_clips = int(NASAL_ALIGN_BUDGET_H * 3600 / ALIGN_SECONDS_PER_CLIP)
NASAL_TARGET = min(wanted, budget_clips)

projected_nasal = BASE_NASAL_RATE * base_vowels + MINED_NASALS_PER_CLIP * NASAL_TARGET
projected_total = base_vowels + MINED_VOWELS_PER_CLIP * NASAL_TARGET
print(f"base corpus     : {base_utterances:,} utterances "
      f"(~{base_vowels:,.0f} vowels, ~{BASE_NASAL_RATE:.1%} nasal)")
print(f"wanted for {NASAL_SHARE_TARGET:.0%}  : {wanted:,} clips")
print(f"time budget     : {budget_clips:,} clips ({NASAL_ALIGN_BUDGET_H} h of alignment)")
if wanted > budget_clips:
    print("  capped by the time budget; raise NASAL_ALIGN_BUDGET_H to lift it")
print(f"projected share : {projected_nasal / max(projected_total, 1):.1%} nasal")
print(f"mined audio     : ~{NASAL_TARGET * 1.4 * 32000 / 1e9:.1f} GB on {TEMP}, "
      f"doubled while bucketing")

# MFA stalls indefinitely on large batches of very short clips. Measured: 400
# clips aligned in 30 s, 4,944 hung for over 25 minutes with no output. Chunking
# is not an optimization here, it is what makes the run finish at all.
CHUNK_GROUPS = 15        # speaker groups per alignment chunk
BUCKET_SIZE = 40         # clips per synthetic speaker group

# michsethowusu is crowdsourced with no speaker labels. Bucketing gives MFA
# something to parallelise over -- one directory of 4,858 files leaves every job
# but one idle. It does NOT make the bucket a speaker: the classifier detects
# these groups by their "_unk" name and normalizes them per utterance instead.

MINED = TEMP / "mined"
(MINED / "corpus").mkdir(parents=True, exist_ok=True)
print(f"model      : {USE_MODEL}")
print(f"dictionary : {USE_DICT}")
print(f"\ntarget     : {NASAL_TARGET:,} nasal-bearing clips")

base corpus     : 106,441 utterances (~521,561 vowels, ~10.3% nasal)
wanted for 17%  : 65,316 clips
time budget     : 60,000 clips (2.5 h of alignment)
  capped by the time budget; raise NASAL_ALIGN_BUDGET_H to lift it
projected share : 16.6% nasal
mined audio     : ~2.7 GB on /kaggle/temp, doubled while bucketing
model      : /kaggle/working/yoruba_model/yoruba_acoustic.zip
dictionary : /kaggle/working/yoruba_model/yoruba.dict

target     : 60,000 nasal-bearing clips


## Mine nasal-bearing clips

Selects only clips whose transcript contains at least one nasal vowel, which is
what shifts the ratio rather than merely adding volume.

Note that WaxalNLP is usable here despite its poor tone coverage: vowel
*identity* depends on the sub-dots (`ẹ`, `ọ`) and the nasal `n`, not on tone
marks.

In [29]:
import io
from collections import Counter

import numpy as np
import soundfile as sf
import pyarrow.parquet as pq

from yoruba_engine.g2p.dictionary import grapheme_to_phonemes

NASAL_PHONES = {"an", "En", "in", "On", "un"}


def nasal_phones_in(text: str):
    """Return the nasal vowels a transcript calls for, or None if unconvertible."""
    try:
        phones = [
            x.symbol.split("_")[0]
            for word in tokenize(text or "", keep_noise=False)
            for x in grapheme_to_phonemes(word)
        ]
    except Exception:
        return None
    return [p for p in phones if p in NASAL_PHONES]


def mine_nasal_clips(paths, source, speaker_field=None, target=10000):
    """Stage clips containing nasal vowels into an MFA corpus."""
    out = MINED / "corpus"
    counts, vocabulary, kept = Counter(), set(), 0

    for path in paths:
        if kept >= target:
            break
        try:
            parquet = pq.ParquetFile(path)
        except Exception as exc:
            print(f"   unreadable {path.name}: {exc}")
            continue

        for batch_index, batch in enumerate(parquet.iter_batches(batch_size=400)):
            if kept >= target:
                break
            data = batch.to_pydict()
            texts = data.get("text") or data.get("sentence") or []
            audios = data.get("audio") or []
            speakers = data.get(speaker_field) if speaker_field else None

            for row, (text, audio) in enumerate(zip(texts, audios)):
                if kept >= target:
                    break
                if not audio or not audio.get("bytes"):
                    continue
                nasals = nasal_phones_in(text)
                if not nasals:
                    continue

                try:
                    samples, rate = sf.read(io.BytesIO(audio["bytes"]),
                                            dtype="float32", always_2d=False)
                except Exception:
                    continue
                if getattr(samples, "ndim", 1) > 1:
                    samples = samples.mean(axis=1)
                duration = len(samples) / rate
                if not (MIN_DURATION_S <= duration <= MAX_DURATION_S):
                    continue
                if rate != 16000:
                    count = int(len(samples) / rate * 16000)
                    samples = np.interp(np.linspace(0, len(samples) - 1, count),
                                        np.arange(len(samples)), samples).astype("float32")
                    rate = 16000

                speaker = f"{source}_{(speakers[row] if speakers else 'unk')}"
                directory = out / speaker
                directory.mkdir(parents=True, exist_ok=True)
                utt_id = f"{source}_{path.stem[-8:]}_{batch_index}_{row}"
                sf.write(str(directory / f"{utt_id}.wav"), samples, rate, subtype="PCM_16")
                (directory / f"{utt_id}.lab").write_text(normalize(text), encoding="utf-8")

                counts.update(nasals)
                vocabulary.update(tokenize(text, keep_noise=False))
                kept += 1

            if kept and kept % 2000 < 400:
                print(f"   {source}: {kept:,} clips")

    print(f"{source}: {kept:,} nasal-bearing clips")
    return kept, counts, vocabulary


mined_vocabulary, nasal_counts, mined_total = set(), Counter(), 0

if USE_MICHSETHOWUSU and michsethowusu_files:
    kept, counts, vocab = mine_nasal_clips(
        michsethowusu_files, "mh", target=int(NASAL_TARGET * 0.85)
    )
    mined_total += kept; nasal_counts += counts; mined_vocabulary |= vocab

if USE_WAXAL and waxal_files:
    kept, counts, vocab = mine_nasal_clips(
        waxal_files, "wx", speaker_field="speaker_id", target=int(NASAL_TARGET * 0.15)
    )
    mined_total += kept; nasal_counts += counts; mined_vocabulary |= vocab

print(f"\nmined {mined_total:,} clips, vocabulary {len(mined_vocabulary):,}")
print("nasal vowels called for by the mined text:")
for phone, count in nasal_counts.most_common():
    print(f"   {phone:<4} {count:>7,}")

   mh: 68 clips
   mh: 120 clips
   mh: 184 clips
   mh: 245 clips
   mh: 304 clips
   mh: 365 clips
   mh: 2,018 clips
   mh: 2,084 clips
   mh: 2,150 clips
   mh: 2,221 clips
   mh: 2,284 clips
   mh: 2,369 clips
   mh: 4,018 clips
   mh: 4,058 clips
   mh: 4,087 clips
   mh: 4,123 clips
   mh: 4,166 clips
   mh: 4,207 clips
   mh: 4,260 clips
   mh: 4,314 clips
   mh: 4,381 clips
   mh: 6,007 clips
   mh: 6,081 clips
   mh: 6,178 clips
   mh: 6,273 clips
   mh: 6,333 clips
   mh: 6,391 clips
   mh: 8,039 clips
   mh: 8,130 clips
   mh: 8,207 clips
   mh: 8,298 clips
   mh: 8,389 clips
   mh: 10,027 clips
   mh: 10,082 clips
   mh: 10,152 clips
   mh: 10,223 clips
   mh: 10,294 clips
   mh: 10,373 clips
   mh: 12,035 clips
   mh: 12,106 clips
   mh: 12,187 clips
   mh: 12,256 clips
   mh: 12,337 clips
   mh: 14,042 clips
   mh: 14,101 clips
   mh: 14,159 clips
   mh: 14,230 clips
   mh: 14,299 clips
   mh: 14,357 clips
   mh: 16,017 clips
   mh: 16,032 clips
   mh: 16,093 clips
   mh

## Bucket into speaker groups

MFA parallelises by speaker. michsethowusu has no speaker labels, so without
bucketing every clip lands in one directory and eleven of twelve jobs sit idle --
the difference between an alignment that finishes and one that does not.

In [30]:
import shutil

source_dir = MINED / "corpus"
bucketed = MINED / "bucketed"
shutil.rmtree(bucketed, ignore_errors=True)
bucketed.mkdir(parents=True)

index = 0
for wav in sorted(source_dir.rglob("*.wav")):
    lab = wav.with_suffix(".lab")
    if not lab.exists():
        continue
    speaker = wav.parent.name
    # Sources with real speaker labels keep them; unlabelled ones are bucketed.
    group = f"{speaker}_{index // BUCKET_SIZE:04d}" if speaker.endswith("_unk") else speaker
    directory = bucketed / group
    directory.mkdir(exist_ok=True)
    shutil.copy2(wav, directory / wav.name)
    shutil.copy2(lab, directory / lab.name)
    index += 1

groups = sorted(d for d in bucketed.iterdir() if d.is_dir())
print(f"{index:,} clips in {len(groups)} speaker groups "
      f"({index / max(len(groups), 1):.0f} per group)")

52,112 clips in 1283 speaker groups (41 per group)


## Extend the dictionary

The mined clips use words the base corpus does not contain, and MFA aborts any
utterance holding an out-of-vocabulary word. The dictionary must therefore cover
both vocabularies, and must use the same phone style as the acoustic model.

In [31]:
combined_vocabulary = sorted(
    {w for w in mined_vocabulary if w.strip()}
    | {t for _, _, text in writer.index_rows for t in tokenize(text, keep_noise=False)}
)

# Rules only. Upload an overrides.tsv and use load_overrides() instead if you
# have hand-corrected pronunciations to preserve.
mined_dictionary = PronunciationDictionary()
mined_dictionary.build(combined_vocabulary)
if not TONE_IN_PHONES:
    mined_dictionary.strip_tone_suffixes()

MINED_DICT = MINED / "combined.dict"
mined_dictionary.write(MINED_DICT)
print(f"dictionary: {len(mined_dictionary.entries):,} entries, "
      f"{len(mined_dictionary.phone_inventory())} phones, "
      f"{len(mined_dictionary.failures)} failures")

# Utterances containing an unmappable word would fail alignment, so drop them.
unmappable = set(mined_dictionary.failures)
removed = 0
if unmappable:
    for lab in bucketed.rglob("*.lab"):
        if any(t in unmappable for t in tokenize(lab.read_text(encoding="utf-8"), keep_noise=False)):
            wav = lab.with_suffix(".wav")
            lab.unlink()
            if wav.exists():
                wav.unlink()
            removed += 1
print(f"removed {removed} utterances containing unmappable words")
print(f"remaining: {len(list(bucketed.rglob('*.wav'))):,}")

dictionary: 9,341 entries, 31 phones, 304 failures
removed 0 utterances containing unmappable words
remaining: 52,112


## Align in chunks

**This must be chunked.** MFA stalls indefinitely on large batches of very short
clips: 400 aligned in 30 seconds, while 4,944 produced no output at all in over
25 minutes. Chunking is what makes the run terminate.

In [32]:
MINED_ALIGNED = MINED / "aligned"
MINED_ALIGNED.mkdir(exist_ok=True)

groups = sorted(d for d in bucketed.iterdir() if d.is_dir())
chunk_root = MINED / "chunks"
done = 0

for chunk_index in range(0, len(groups), CHUNK_GROUPS):
    batch = groups[chunk_index:chunk_index + CHUNK_GROUPS]
    staging = chunk_root / f"c{chunk_index // CHUNK_GROUPS:03d}"
    shutil.rmtree(staging, ignore_errors=True)
    staging.mkdir(parents=True)
    for group in batch:
        shutil.copytree(group, staging / group.name)

    try:
        run_mfa(
            ["align", staging, MINED_DICT, USE_MODEL, staging / "out",
             "--num_jobs", NUM_JOBS, "--clean"],
            f"chunk {chunk_index // CHUNK_GROUPS + 1} of "
            f"{(len(groups) + CHUNK_GROUPS - 1) // CHUNK_GROUPS}",
        )
    except Exception as exc:
        print(f"   chunk failed, continuing: {exc}")
        shutil.rmtree(staging, ignore_errors=True)
        continue

    for textgrid in (staging / "out").rglob("*.TextGrid"):
        shutil.copy2(textgrid, MINED_ALIGNED / textgrid.name)
    shutil.rmtree(staging, ignore_errors=True)

    done = len(list(MINED_ALIGNED.glob("*.TextGrid")))
    print(f"   running total: {done:,} TextGrids")

print(f"\naligned {done:,} mined clips")

=== chunk 1 of 86 ===
/kaggle/temp/mfa-env/bin/mfa align /kaggle/temp/mined/chunks/c000 /kaggle/temp/mined/combined.dict /kaggle/working/yoruba_model/yoruba_acoustic.zip /kaggle/temp/mined/chunks/c000/out --num_jobs 4 --clean

 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
 INFO     Found 15 speakers across 600 files, average number of utterances per  
          speaker: 40.0                                                         
 INFO     Initializing multiprocessing jobs...                                  
 INFO     Normalizing text...                                                   
 INFO     Generating MFCCs...                                                   
 INFO     Calculating CMVN...                                                   
 INFO     Generating final features...                                          
 INFO     Creating corpus split...          

## Train the vowel classifier

Trained on the base corpus plus the mined clips, with **class weighting**.

Weighting lowers overall accuracy while raising macro F1 -- the right trade here,
because this model is read for the probability it assigns the *expected* vowel,
not for its top guess. A model that is confidently right about `a` and never
predicts `an` scores well on accuracy and is useless for catching `òkan`.

Evaluation splits by **speaker**. Formants scale with vocal tract length and the
features are normalized per speaker, so a random split would let the model
memorize voices and report an accuracy it cannot reproduce on a learner.

In [33]:
import random

import numpy as np
from sklearn.metrics import accuracy_score, classification_report, f1_score

from yoruba_engine.pitch.extractor import is_vowel_phone
from yoruba_engine.alignment.textgrid import alignment_from_textgrid
from yoruba_engine.segmental.classifier import (
    VowelClassifier, build_estimator, combined_feature_names, vowel_features,
)
from yoruba_engine.segmental.formants import FormantExtractor, normalize_vowel_space
from yoruba_engine.segmental.mfcc import MFCCExtractor, cepstral_mean_variance, normalize_vector

# en and on are not Yoruba vowels; they appear only in alignments produced
# before the grapheme-to-phoneme rules were corrected.
ARTEFACT = {"en": "e", "on": "o"}

formant_extractor = FormantExtractor()
mfcc_extractor = MFCCExtractor()

audio_index = {w.stem: w for w in bucketed.rglob("*.wav")}
audio_index.update({w.stem: w for w in CORPUS.rglob("*.wav")})


def is_real_speaker(name: str) -> bool:
    """Whether a corpus directory holds one person rather than a bucket.

    Bucketed pseudo-speakers ("michsethowusu_unk00000", "mh_unk_0000") hold
    clips from 40-50 different people. Anything keyed on speaker identity has
    to know the difference.
    """
    return "_unk" not in name


# Cepstral mean normalization removes vocal tract length and channel response,
# but only when the group it averages over is one person AND large enough that
# the mean does not track the vowels themselves. Measured on this corpus, with
# group size swept on a single speaker so only sample size varies:
#
#   group size    2      3      5     10     20     40     80   none
#   macro F1   0.227  0.282  0.300  0.342  0.362  0.390  0.375  0.342
#
# Below about 10 vowels it is actively destructive -- the mean IS the vowel --
# and it only clears the do-nothing baseline from roughly 20 up.
#
# Strangers break it regardless of size. On unlabelled michsethowusu clips
# (median 3 vowels each) larger buckets are monotonically better and doing
# nothing wins outright:
#
#   bucket      1 clip  5   20   40   100   all(= no centering)
#   macro F1     0.387 0.472 0.520 0.530 0.541  0.549
#
# So: normalize only real speakers with enough material, and leave everything
# else alone. Note this is what the pre-bucketing notebook did by accident for
# michsethowusu -- one giant directory made its CMVN a global no-op -- which is
# why that run scored better on nasals than the per-utterance version did.
MIN_CMVN_VOWELS = 40

sources = [(ALIGNED, "corpus"), (MINED_ALIGNED, "mined")]
pending = {}
processed = skipped = 0

for directory, label in sources:
    textgrids = sorted(directory.rglob("*.TextGrid"))
    print(f"{label}: {len(textgrids):,} TextGrids")

    for textgrid in textgrids:
        audio = audio_index.get(textgrid.stem)
        if audio is None:
            skipped += 1
            continue
        speaker = audio.parent.name
        try:
            alignment = alignment_from_textgrid(textgrid)
            formants = formant_extractor.analyse(audio)
            times, frames = mfcc_extractor.analyse(audio)
        except Exception:
            skipped += 1
            continue

        vowels = [p for p in alignment.phones if is_vowel_phone(p.label)]
        measured = []
        for phone in vowels:
            formant_values = formant_extractor.measure_segment(formants, phone.start_s, phone.end_s)
            cepstral_values = mfcc_extractor.measure_segment(times, frames, phone.start_s, phone.end_s)
            if formant_values and cepstral_values:
                measured.append((phone.label, formant_values, cepstral_values))
        if not measured:
            continue

        speaker_f1, speaker_f2 = normalize_vowel_space([m[1] for m in measured])
        for label_text, formant_values, cepstral_values in measured:
            base = label_text.split("_")[0]
            base = ARTEFACT.get(base, base)
            pending.setdefault(speaker, []).append(
                (vowel_features(formant_values, speaker_f1, speaker_f2),
                 list(cepstral_values), base, speaker)
            )

        processed += 1
        if processed % 1000 == 0:
            print(f"   {processed:,} utterances")

# No cepstral normalization, and no feature scaling either.
#
# Whatever is done here must also be doable on a single learner recording,
# because the engine applies the identical transform before calling the model.
# It cannot be: one utterance holds a handful of vowels, and the table above
# shows normalizing over that few is worse than doing nothing. Any global
# constant fitted here (a corpus-wide mean, a corpus-wide standard deviation)
# has the same problem in a quieter way -- it has to be shipped alongside the
# model and applied identically at inference, and when it is not, the boosted
# tree's thresholds refer to a scale the incoming features are not on.
#
# Measured cost of dropping it entirely: macro F1 0.605 -> 0.593. Measured cost
# of getting it wrong at inference: the classifier scored the correct vowel at
# p = 0.002 on real recordings of okun and produced the wrong verdict on 8 of
# 11 of them.
rows, labels, speakers = [], [], []
for group, entries in pending.items():
    for formant_row, cepstral_row, base, speaker in entries:
        rows.append(formant_row + list(cepstral_row))
        labels.append(base)
        speakers.append(speaker)

real = sum(1 for s in set(speakers) if is_real_speaker(s))
print(f"\ncollected {len(rows):,} vowels from {processed:,} utterances ({skipped} skipped)")
print(f"speakers: {real:,} real, {len(set(speakers)) - real:,} bucketed")

# Cache the measurements BEFORE normalization. Extraction needs the audio and
# the alignments, which only exist inside this session; everything after it --
# the normalization rule, the split, the class weighting, the thresholds -- is
# arithmetic on this array. Saving the raw form means those can all be retried
# locally in seconds, instead of each attempt costing a Kaggle session.
#
# The formant block is already per-utterance normalized, which is independent
# of the cepstral question, so it is stored as computed.
FEATURES = OUTPUT / "vowel_features.npz"
np.savez_compressed(
    FEATURES,
    formants=np.asarray([f for entries in pending.values() for f, _, _, _ in entries],
                        dtype="float32"),
    cepstra=np.asarray([c for entries in pending.values() for _, c, _, _ in entries],
                       dtype="float32"),
    labels=np.asarray([b for entries in pending.values() for _, _, b, _ in entries]),
    speakers=np.asarray([s for entries in pending.values() for _, _, _, s in entries]),
    feature_names=np.asarray(list(combined_feature_names())),
)
print(f"cached RAW features -> {FEATURES} ({FEATURES.stat().st_size / 1e6:.0f} MB)")
print("  download this file; the classifier can then be retrained locally "
      "without another Kaggle run")

corpus: 104,961 TextGrids
   1,000 utterances
   2,000 utterances
   3,000 utterances
   4,000 utterances
   5,000 utterances
   6,000 utterances
   7,000 utterances
   8,000 utterances
   9,000 utterances
   10,000 utterances
   11,000 utterances
   12,000 utterances
   13,000 utterances
   14,000 utterances
   15,000 utterances
   16,000 utterances
   17,000 utterances
   18,000 utterances
   19,000 utterances
   20,000 utterances
   21,000 utterances
   22,000 utterances
   23,000 utterances
   24,000 utterances
   25,000 utterances
   26,000 utterances
   27,000 utterances
   28,000 utterances
   29,000 utterances
   30,000 utterances
   31,000 utterances
   32,000 utterances
   33,000 utterances
   34,000 utterances
   35,000 utterances
   36,000 utterances
   37,000 utterances
   38,000 utterances
   39,000 utterances
   40,000 utterances
   41,000 utterances
   42,000 utterances
   43,000 utterances
   44,000 utterances
   45,000 utterances
   46,000 utterances
   47,000 utteran

In [34]:
import hashlib

label_set = sorted(set(labels))
position = {label: i for i, label in enumerate(label_set)}
X = np.asarray(rows, dtype=float)
y = np.asarray([position[l] for l in labels], dtype=int)

# The held-out set is built only from directories that hold ONE person.
# A bucket is 40-50 strangers, so holding one out is not holding out a voice --
# the same person's other clips sit in a neighbouring bucket, in training.
#
# Membership is decided by hashing the speaker id, not by shuffling the list
# that this run happened to produce. Changing MAX_HOURS changes which speakers
# appear, which would reshuffle a list-based split and silently move speakers
# between train and test -- exactly what made the 1 h and 48 h runs
# incomparable (72.5% on 38 speakers vs 67.5% on 122 different ones). Hashing
# puts a given speaker on the same side of the split in every run.
HOLDOUT_PERCENT = 25

def in_holdout(name: str) -> bool:
    digest = hashlib.md5(name.encode("utf-8")).hexdigest()
    return int(digest[:8], 16) % 100 < HOLDOUT_PERCENT

unique_speakers = sorted(set(speakers))
real_speakers = [s for s in unique_speakers if is_real_speaker(s)]
holdout = {s for s in real_speakers if in_holdout(s)}
train_idx = [i for i, s in enumerate(speakers) if s not in holdout]
test_idx = [i for i, s in enumerate(speakers) if s in holdout]

counts = Counter(labels)
print("class balance:", {k: counts[k] for k in label_set})
print(f"train {len(train_idx):,} / held-out {len(test_idx):,} "
      f"across {len(holdout)} of {len(real_speakers)} real speakers "
      f"({len(unique_speakers) - len(real_speakers)} buckets are training-only)\n")

if not test_idx:
    raise SystemExit(
        "Nothing held out. Every speaker directory looks like a bucket, so no "
        "honest evaluation is possible -- enable a source with speaker labels "
        "(USE_OPENSLR, USE_LOCAL_TTS or USE_WAXAL)."
    )

by_source = Counter(s.split("_")[0] for s in holdout)
print("held-out speakers by source:", dict(by_source))

model = build_estimator(balanced=True)
model.fit(X[train_idx], y[train_idx])
predicted = model.predict(X[test_idx])

accuracy = accuracy_score(y[test_idx], predicted)
macro = f1_score(y[test_idx], predicted, average="macro", zero_division=0)
print(f"held-out accuracy : {accuracy * 100:.1f}%")
print(f"macro F1          : {macro:.3f}   (local reference: 0.525)\n")
print(classification_report(y[test_idx], predicted, labels=list(range(len(label_set))),
                            target_names=label_set, digits=3, zero_division=0))

class balance: {'E': 60519, 'En': 447, 'O': 63937, 'On': 31900, 'a': 169470, 'an': 24321, 'e': 61632, 'i': 184901, 'in': 24367, 'o': 91926, 'u': 51148, 'un': 41055}
train 771,430 / held-out 34,193 across 17 of 64 real speakers (3266 buckets are training-only)

held-out speakers by source: {'local': 5, 'openslr': 8, 'wx': 3, 'waxal': 1}
held-out accuracy : 71.0%
macro F1          : 0.561   (local reference: 0.525)

              precision    recall  f1-score   support

           E      0.585     0.778     0.668      1924
          En      0.750     0.207     0.324        29
           O      0.562     0.676     0.614      2287
          On      0.398     0.461     0.427       912
           a      0.918     0.905     0.912      7970
          an      0.297     0.414     0.346       758
           e      0.607     0.664     0.634      3706
           i      0.849     0.692     0.762      7565
          in      0.231     0.324     0.270       528
           o      0.831     0.684     0.7

In [35]:
# Nasal F1 is the number this whole exercise exists to move.
NASAL_LABELS = ["an", "En", "in", "On", "un"]
REFERENCE = {"an": 0.358, "En": 0.118, "in": 0.297, "On": 0.483, "un": 0.488}

per_class = f1_score(y[test_idx], predicted, labels=list(range(len(label_set))),
                     average=None, zero_division=0)
print(f"{'nasal':<7}{'this run':>10}{'local ref':>11}{'change':>9}")
for nasal in NASAL_LABELS:
    if nasal not in position:
        continue
    value = per_class[position[nasal]]
    reference = REFERENCE[nasal]
    print(f"{nasal:<7}{value:>10.3f}{reference:>11.3f}{value - reference:>+9.3f}")

# An F1 computed on a handful of samples is noise, not a measurement. En in
# particular is rare enough that it has previously been scored on 13 samples.
MIN_RELIABLE_SUPPORT = 30
test_counts = Counter(label_set[i] for i in y[test_idx])
thin = [n for n in NASAL_LABELS if 0 < test_counts.get(n, 0) < MIN_RELIABLE_SUPPORT]
if thin:
    print("\nheld-out support below "
          f"{MIN_RELIABLE_SUPPORT}, treat these F1 values as noise: "
          + ", ".join(f"{n} ({test_counts[n]})" for n in thin))

final = build_estimator(balanced=True)
final.fit(X, y)
classifier = VowelClassifier(
    model=final, labels=label_set,
    metadata={
        "training_vowels": int(len(y)),
        "speakers": len(unique_speakers),
        "holdout_accuracy": round(float(accuracy), 4),
        "macro_f1": round(float(macro), 4),
        "class_balanced": True,
        "uses_mfcc": True,
        # The engine reads these two and must be able to reproduce them.
        "cepstral_normalization": "none",
        "audio_sample_rate": 16000,
        "mined_clips": int(mined_total),
        "features": list(combined_feature_names()),
    },
)
classifier.save(OUTPUT / "vowel_classifier.joblib")
print(f"\nsaved to {OUTPUT / 'vowel_classifier.joblib'}")

nasal    this run  local ref   change
an          0.346      0.358   -0.012
En          0.324      0.118   +0.206
in          0.270      0.297   -0.027
On          0.427      0.483   -0.056
un          0.496      0.488   +0.008

held-out support below 30, treat these F1 values as noise: En (29)

saved to /kaggle/working/yoruba_model/vowel_classifier.joblib


## Using it

Copy `vowel_classifier.joblib` into the project's `models/`. `AssessmentPipeline`
loads it automatically and falls back to omission-only detection when it is
absent, so nothing else needs changing.

Then re-check the operating point locally, because the thresholds were calibrated
against a different model:

```bash
python scripts/demo_assess.py          # end-to-end sanity check
```

`SegmentalConfig.suspect_probability` (0.02) was set so that ~3% of correctly
spoken words are queried. If the mined model separates better, that threshold can
be raised to catch more substitutions at the same false-query rate — but measure
it rather than assuming, since raising it also queries more correct speech.

**The number to watch is the nasal F1 table above.** If `an` and `in` have moved
well past 0.36 and 0.30, the `òkun`/`òkàn` discrimination should have enough
confidence to report a verdict, which it did not have locally.